In [2]:
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

True

In [3]:
df = pd.read_csv("auto.csv")

In [4]:
df.head()

,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495
0,3,?,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500
1,1,?,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500
2,2,164,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950
3,2,164,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450
4,2,?,audi,gas,std,two,sedan,fwd,front,99.8,...,136,mpfi,3.19,3.40,8.5,110,5500,19,25,15250


In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize the Model
# Temperature 0.9 is great for creative naming tasks
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

# 2. Define the Prompt Template
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# 3. Create the Chain using LCEL (The Pipe Operator)
# This replaces the LLMChain class entirely
chain = prompt | llm

# 4. Run the chain
product = "Queen Size Sheet Set"

# Use .invoke() instead of .run()
response = chain.invoke({"product": product})

print(response.content)

Royal Comfort Bedding Co.


In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize Model
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9)

# 2. Define Prompts
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following company: {company_name}"
)

# 3. Create the Sequential Chain (The Modern way)
# We use StrOutputParser() to convert the first AI message into a string for the second prompt
chain = (
    {"company_name": first_prompt | llm | StrOutputParser()} 
    | second_prompt 
    | llm 
    | StrOutputParser()
)

# 4. Execute
product = "Queen Size Sheet Set"
result = chain.invoke({"product": product})

print(result)

Regal Rest Linens offers luxurious and high-quality bedding and linens for a comfortable and elegant sleep experience every night.


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-5.2", temperature=0.9)

# 1. Define the Prompts
first_prompt = ChatPromptTemplate.from_template("Translate to English:\n\n{Review}")
second_prompt = ChatPromptTemplate.from_template("Summarize in 1 sentence:\n\n{English_Review}")
third_prompt = ChatPromptTemplate.from_template("What language is this:\n\n{Review}")
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up to this summary in this language:\n\nSummary: {summary}\n\nLanguage: {language}"
)

# 2. Build the LCEL Chain
# We use dictionaries to "pass through" variables from previous steps
chain = (
    {"Review": RunnablePassthrough()}
    # Step 1: Translate
    | RunnablePassthrough.assign(English_Review=first_prompt | llm | StrOutputParser())
    # Step 2: Summarize (using the translation from step 1)
    | RunnablePassthrough.assign(summary=second_prompt | llm | StrOutputParser())
    # Step 3: Detect Language (using the original review)
    | RunnablePassthrough.assign(language=third_prompt | llm | StrOutputParser())
    # Step 4: Final Follow-up
    | RunnablePassthrough.assign(followup_message=fourth_prompt | llm | StrOutputParser())
)

# 3. Execute
review = "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?" # Example review
result = chain.invoke(review)

# The result is a dictionary containing all assigned keys
print(f"Summary: {result['summary']}")
print(f"Follow-up: {result['followup_message']}")

Summary: The reviewer says the product tastes mediocre with unstable foam compared to store-bought versions and suspects it may be an old or counterfeit batch.
Follow-up: Merci pour votre retour. Nous sommes désolés d’apprendre que le goût vous a semblé médiocre et que la mousse n’était pas stable.

Pour vérifier ce qui s’est passé, pourriez-vous nous indiquer :
- la date de péremption (DDM/DLC) et le numéro de lot figurant sur l’emballage,
- où et quand vous l’avez acheté (vendeur/plateforme),
- si possible, une photo de l’étiquette et du produit.

Nous pourrons ainsi confirmer l’authenticité du produit et, si un lot présente un défaut ou s’il a été mal stocké, vous proposer une solution (remplacement ou remboursement selon le cas).


In [14]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 1. Define the specialized chains (Destinations)
physics_chain = ChatPromptTemplate.from_template(physics_template) | llm | StrOutputParser()
math_chain = ChatPromptTemplate.from_template(math_template) | llm | StrOutputParser()
history_chain = ChatPromptTemplate.from_template(history_template) | llm | StrOutputParser()
cs_chain = ChatPromptTemplate.from_template(computerscience_template) | llm | StrOutputParser()
default_chain = ChatPromptTemplate.from_template("{input}") | llm | StrOutputParser()

# 2. Define the Routing Logic
def route(info):
    topic = info["topic"].lower().strip()
    # --- Custom Verbose Output ---
    print(f"\n> [ROUTER] Selected Destination: {topic}")
    # -----------------------------
    if "physics" in topic:
        return physics_chain
    elif "math" in topic:
        return math_chain
    elif "history" in topic:
        return history_chain
    elif "computer science" in topic:
        return cs_chain
    else:
        return default_chain

# 3. Create the Router Prompt
# We ask the LLM to simply categorize the input
router_prompt = ChatPromptTemplate.from_template(
    "Given the user question, classify it as 'physics', 'math', 'history', or 'computer science'. "
    "If it fits none, say 'default'.\n\nQuestion: {input}\n\nCategory:"
)

# 4. Build the Final Chain
full_chain = (
    {"topic": router_prompt | llm | StrOutputParser(), "input": lambda x: x["input"]}
    | RunnableLambda(route)
)

# 5. Execute
result = full_chain.invoke({"input": "What is black body radiation?"})
print(result)


> [ROUTER] Selected Destination: physics
Black body radiation is the electromagnetic radiation emitted by a perfect absorber of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation across the entire electromagnetic spectrum. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power.


In [18]:
result = full_chain.invoke({"input": "What is 2 + 2?"})
print(result)


> [ROUTER] Selected Destination: math
The answer to 2 + 2 is 4.


In [19]:
result = full_chain.invoke({"input": "Why does every cell in our body contain DNA?"})
print(result)


> [ROUTER] Selected Destination: biology
Every cell in our body contains DNA because DNA carries the genetic information that determines the characteristics and functions of an organism. DNA contains the instructions for building and maintaining an organism, including the proteins that are essential for cell structure and function. This genetic information is passed down from parent to offspring and is essential for the growth, development, and functioning of all cells in the body. Having DNA in every cell ensures that the genetic information is preserved and can be used to carry out the necessary processes for life.
